# **Cleaning and Analysing Retail Sales Data Using PostgreSQL**

![project banner](images/banner_image.png)

# Cleaning a PostgreSQL Database
![Clean PostgreSQL Database](Project_Image.jpeg)

In this project, you will work with data from a hypothetical Super Store to challenge and enhance your SQL skills in data cleaning. This project will engage you in identifying top categories based on the highest profit margins and detecting missing values, utilizing your comprehensive knowledge of SQL concepts.

## Data Dictionary:

### `orders`:
| Column | Definition | Data type | Comments |
|--------|------------|-----------|----------|
| `row_id`| Unique Record ID | `INTEGER` |
| `order_id` | Identifier for each order in table | `TEXT` | Connects to `order_id` in `returned_orders` table |
| `order_date` | Date when order was placed | `TEXT` |
| `market` | Market order_id belongs to | `TEXT` |
| `region` | Region Customer belongs to | `TEXT` | Connects to `region` in `people` table |
| `product_id` | Identifier of Product bought | `TEXT` | Connects to `product_id` in `products` table |
| `sales` | Total Sales Amount for the Line Item | `DOUBLE PRECISION` |
| `quantity` | Total Quantity for the Line Item | `DOUBLE PRECISION` |
| `discount` | Discount applied for the Line Item | `DOUBLE PRECISION` |
| `profit` | Total Profit earned on the Line Item | `DOUBLE PRECISION` |

### `returned_orders`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `returned`| Yes values for Order / Line Item Returned | `TEXT` |
| `order_id` | Identifier for each order in table | `TEXT` |
| `market` | Market order_id belongs to | `TEXT` |

### `people`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `person`| Name of Salesperson credited with Order | `TEXT` |
| `region` | Region Salesperson in operating in | `TEXT` |

### `products`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `product_id`| Unique Identifier for the Product | `TEXT` |
| `category` | Category Product belongs to | `TEXT` |
| `sub_category` | Sub Category Product belongs to | `TEXT` |
| `product_name` | Detailed Name of the Product | `TEXT` |

As you can see in the Data Dictionary above, date fields have been written to the `orders` table as `TEXT` and numeric fields like sales, profit, etc. have been written to the `orders` table as `Double Precision`. You will need to take care of these types in some of the queries. This project is an excellent opportunity to apply your SQL skills in a practical setting and gain valuable experience in data cleaning and analysis. Good luck, and happy querying!

## **Task 1: Identify the top five products from each category with highest profit margin**
Find the top 5 products from each category based on highest total sales. The output should be sorted by category in ascending order and by sales in descending order within each category, i.e. within each category product with highest margin should sit on the top. Save the query as top_five_products_each_category, containing the following columns:
- category
- product_name
- product_total_sales (rounded to two decimal places)
- product_total_profit (rounded to two decimal places)
- product_rank 

### **Solution to Task 1**

Build a query to return the top 5 products in each of the categories.
- There are 3 categories.
- Top 5 products for each of the categories are calculated based on highest total sales.
- So, there should be 15 rows returned, five for each category.

Aggregating and ranking
- To start, you can SELECT * and use a subquery that selects the required columns from the orders and products tables. the query with Windows Function as a sub query and in the outer query use WHERE to only retain wanted number of rows
- Inside the subquery, select the category and product_name columns from the products table.
- Calculate product_total_profit and product_total_sales by taking the SUM() of the profit and sales columns from the orders table; CASTing to NUMERIC to successfully perform the calculation.
- You can wrap your calculations inside a call of ROUND() to round the results to 2 decimal places.
- To rank products by highest sales amounts, use the RANK() function with OVER(), PARTITIONing BY category and ORDERing BY the SUM() of sales in DESCending order.
- Perform an INNER JOIN to combine the orders and products tables, linking by product_id.
- Remember to use GROUP BY on the columns that you are not performing a calculation on.
- Remember to give your subquery a name, such as tmp, to avoid a syntax error.
- Outside of the subquery, use a WHERE keyword to to filter the product_rank column, only retaining values below 6 so you have the top five products per category.

In [14]:
-- top_five_products_each_category
SELECT
    category,
    product_name,
    product_total_sales,
    product_total_profit,
    product_rank
FROM
(
    SELECT
        p.category,
        p.product_name,
	    ROUND(SUM(CAST(o.sales AS NUMERIC)), 2) AS product_total_sales,
	    ROUND(SUM(CAST(o.profit AS NUMERIC)), 2) AS product_total_profit,
	    RANK() OVER(PARTITION BY p.category ORDER BY SUM(CAST(o.sales AS NUMERIC)) DESC) AS product_rank
    FROM orders o
    INNER JOIN products p
        ON o.product_id = p.product_id
    GROUP BY
        p.category,
        p.product_name
) tmp
WHERE product_rank <= 5
ORDER BY
    category ASC,
    product_total_sales DESC;

,category,product_name,product_total_sales,product_total_profit,product_rank
0,Furniture,"Hon Executive Leather Armchair, Adjustable",58193.48,5997.25,1
1,Furniture,"Office Star Executive Leather Armchair, Adjust...",51449.80,4925.80,2
2,Furniture,"Harbour Creations Executive Leather Armchair, ...",50121.52,10427.33,3
3,Furniture,"SAFCO Executive Leather Armchair, Black",41923.53,7154.28,4
4,Furniture,"Novimex Executive Leather Armchair, Adjustable",40585.13,5562.35,5
5,Office Supplies,"Eldon File Cart, Single Width",39873.23,5571.26,1
6,Office Supplies,"Hoover Stove, White",32842.60,-2180.63,2
7,Office Supplies,"Hoover Stove, Red",32644.13,11651.68,3
8,Office Supplies,"Rogers File Cart, Single Width",29558.82,2368.82,4
9,Office Supplies,"Smead Lockers, Industrial",28991.66,3630.44,5


## **Task 2: Calculating missing quantity values**
Calculate quantity values where this field is missing.

Calculate the quantity for orders with missing values in the quantity column by determining the unit price for each product_id using available order data, considering relevant pricing factors such as discount, market, or region. Then, use this unit price to estimate the missing quantity values. The calculated values should be stored in the calculated_quantity column. Save query output as impute_missing_values, containing the following columns:
- product_id
- discount
- market
- region
- sales
- quantity
- calculated_quantity (rounded to zero decimal places)

### **Solution to Task 2**

General approach
- Find missing quantities: look for orders where the quantity is missing and store their details (product_id, discount, market, region, sales).
- Find other orders of the same product_id, discount, market, and region where quantity is available, then calculate the unit price (sales / quantity).
- Estimate missing quantities: use the unit price to estimate (impute) the missing quantity values by dividing sales by the unit price, rounding to the nearest whole number.

Structuring the query
- The query should have two CTEs: one to find all rows in orders where quantity values are missing, and another to calculate the unit price for each product.
- You can then calculate calculated_quantity and return all required columns for the output: product_id, discount, market, region, sales, quantity, calculated_quantity.

The first CTE: missing
- Create a CTE called missing to get all rows in the orders table with missing values for the quantity column.
- It should SELECT all columns required for the final output: product_id, discount, market, region, sales, quantity, calculated_quantity.
- Filter for rows where quantity is NULL.
- Once you have found the related information and calculate their unit price in 2nd CTE.
- In the main query join data from 1st CTE and 2nd CTEon product_id
  - Get all details from 1st CTE and calculate missing quantity by dividing sales from 1st CTE by unit_price from 2nd CTE

The second CTE: unit_prices
- In the second CTE, called unit_prices find product_id and alias a new column called unit_price, which will contain the price for a single quantity of each product.
- To perform the unit_price calculation you'll need to divide sales by quantity and wrap this in a CAST() function, converting to NUMERIC data type.
- The data should be SELECTed from the orders table, and then joined to the first CTE, missing.
- You need to join on product_id and discount to get related data only.

Finishing the query
- With both CTEs complete, you can SELECT all DISTINCT rows FROM the first CTE, missing.
- You then need add the calculated_quantity column to the output.
- To do this, you need to CAST() the sales column from the missing CTE as NUMERIC and divide this by the unit_price column from the unit_prices CTE.
- As you want calculated_quantity as whole numbers, you can ROUND() the results to 0 decimal places.
- Finish by using an INNER JOIN to the unit_prices CTE, joining on product_id.

In [16]:
-- impute_missing_values
WITH missing AS (
	SELECT product_id,
		discount, 
		market,
		region,
		sales,
		quantity
	FROM orders 
	WHERE quantity IS NULL
), 

unit_prices AS (SELECT o.product_id,
	CAST(o.sales / o.quantity AS NUMERIC) AS unit_price
FROM orders o
RIGHT JOIN missing AS m 
	ON o.product_id = m.product_id
	AND o.discount = m.discount
WHERE o.quantity IS NOT NULL
)

SELECT DISTINCT m.*,
	ROUND(CAST(m.sales AS NUMERIC) / up.unit_price,0) AS calculated_quantity
FROM missing AS m
INNER JOIN unit_prices AS up
	ON m.product_id = up.product_id;

,product_id,discount,market,region,sales,quantity,calculated_quantity
0,FUR-ADV-10000571,0.00,EMEA,EMEA,438.960,NaN,4
1,FUR-ADV-10004395,0.00,EMEA,EMEA,84.120,NaN,2
2,FUR-BO-10001337,0.15,US,West,308.499,NaN,3
3,TEC-STA-10003330,0.00,Africa,Africa,506.640,NaN,2
4,TEC-STA-10004542,0.00,Africa,Africa,160.320,NaN,4
